# 05 — Experimento 2: detección con el modelo de lenguaje

¿Distingue el modelo de lenguaje, solo con los datos de la ventana, las anomalías reales del comportamiento normal? ¿Ayuda darle una descripción del sistema (N1 frente a N0)? (apartado 5.2 de la memoria).

**Prompts.** Los textos de los prompts, los nombres que ve el modelo y tu referencia están protegidos por el acuerdo de confidencialidad: viven en la carpeta `privado/` de Drive y nunca en el repositorio. Los prompts se escriben con los nombres cortos (S01ERR…) y `llm_lab.prompts` pone los nombres reales al montar cada mensaje, a partir de `privado/nombres_senales.json`.

| Sección | Qué hace | Entorno |
|---|---|---|
| 1 | Entorno | CPU |
| 2 | Comprobación: los prompts en ficheros montan exactamente los mensajes de la memoria | CPU |
| 3 | Entradas: las 29 ventanas de desarrollo y las 40 de prueba con tu referencia (sin tus notas) | CPU |
| 4 | Ejecuciones: copia de las ya hechas y, si se quiere repetir, ejecución de un modelo | GPU (solo 4.3) |

El análisis y las cifras de la memoria están en `06_experimento_2_resultados`.

**Confidencial:** este notebook no muestra textos de prompts ni respuestas del modelo; solo cifras y nombres cortos. No subas al repositorio versiones con otras salidas.

## 1. Entorno

En Colab, monta Google Drive y clona el repositorio para tener acceso a `src/`. En local, busca la carpeta `src/` subiendo desde el directorio actual.

In [ ]:
# --- Entorno: localiza src/ y, en Colab, monta Google Drive ---
import os
import subprocess
import sys

EN_COLAB = "google.colab" in sys.modules
URL_REPOSITORIO = "https://github.com/Williams-MP/TFM_VIU_2026.git"
RUTA_REPO_COLAB = "/content/TFM_VIU_2026"

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir(RUTA_REPO_COLAB):
        # Mientras el repositorio sea privado hace falta un token de GitHub.
        # Si es público, basta con dejar el campo vacío.
        from getpass import getpass
        token = getpass("Token de GitHub (vacío si el repositorio es público): ")
        url = URL_REPOSITORIO.replace("https://", f"https://{token}@") if token else URL_REPOSITORIO
        subprocess.run(["git", "clone", "-q", url, RUTA_REPO_COLAB], check=True)
    ruta_src = f"{RUTA_REPO_COLAB}/src"
else:
    # Ejecución local: se sube por las carpetas hasta encontrar src/.
    directorio = os.getcwd()
    while not os.path.isdir(os.path.join(directorio, "src")):
        padre = os.path.dirname(directorio)
        if padre == directorio:
            raise FileNotFoundError("No se encuentra la carpeta src/ del repositorio.")
        directorio = padre
    ruta_src = os.path.join(directorio, "src")

if ruta_src not in sys.path:
    sys.path.append(ruta_src)

import glob
import shutil

import pandas as pd

from config import RUTA_PREPARADOS, IDX_CONTADORES, IDX_GAP, RUTA_EXPERIMENTO_2_PRIVADO
from llm_lab import exp2
from llm_lab.prompts import Prompts
from llm_lab.ventanas import FuenteVentanas
from llm_lab.verificacion import verificar_ejecucion

RUTA_EJECUCIONES_37 = f"{RUTA_EXPERIMENTO_2_PRIVADO}/ejecuciones_37"   # ejecuciones de la memoria (Qwen2.5-7B)
RUTA_ENTRADAS = f"{RUTA_EXPERIMENTO_2_PRIVADO}/entradas"               # ventanas y referencia del autor
RUTA_EJECUCIONES = f"{RUTA_EXPERIMENTO_2_PRIVADO}/ejecuciones"         # resto de ejecuciones (otros modelos)

prompts = Prompts()                                                     # lee privado/prompts y nombres_senales.json
fuente = FuenteVentanas(RUTA_PREPARADOS, IDX_CONTADORES, IDX_GAP)
display(prompts.listar_variantes())

## 2. Comprobación de los prompts

Las ejecuciones del Experimento 2 de la memoria (`fase1_*` en desarrollo y `fase2_*` en prueba) guardaron el mensaje exacto de cada llamada. Aquí se vuelve a montar cada uno con los ficheros de `privado/prompts` y la ventana real, y se comprueba que son **idénticos**.

**2.1 (una sola vez, solo el autor):** copia esas ejecuciones del repositorio de trabajo a `privado/experimento_2/ejecuciones_37/`. Lee la carpeta antigua y no la modifica. Si ya están copiadas, no hace nada.

In [ ]:
# --- 2.1 Copia (una sola vez) de las ejecuciones de la memoria a la carpeta privada del experimento ---
ORIGEN_37 = "/content/drive/MyDrive/TFM_VIU/export_colab_split_estratificado/h2/pruebas_finales"   # solo lectura

os.makedirs(RUTA_EJECUCIONES_37, exist_ok=True)
copiados = 0
for ruta in sorted(glob.glob(f"{ORIGEN_37}/fase[12]_*.csv") + glob.glob(f"{ORIGEN_37}/fase[12]_*.json")):
    destino = f"{RUTA_EJECUCIONES_37}/{os.path.basename(ruta)}"
    if not os.path.exists(destino) or os.path.getmtime(ruta) > os.path.getmtime(destino):   # nuevo o actualizado
        shutil.copy2(ruta, destino)
        copiados += 1
print(f"Copiados: {copiados} | en la carpeta: {sorted(os.listdir(RUTA_EJECUCIONES_37))}")

In [ ]:
# --- 2.2 Cada mensaje guardado frente al que montan los ficheros de privado/prompts ---
resumen = []
for ruta_json in sorted(glob.glob(f"{RUTA_EJECUCIONES_37}/fase[12]_*.json")):
    r = verificar_ejecucion(prompts, fuente, ruta_json, ruta_json[:-5] + ".csv")
    resumen.append({"ejecucion": os.path.basename(ruta_json)[:-5], "llamadas": len(r),
                    "ventana_igual": int(r["ventana_igual"].sum()), "mensajes_iguales": int(r["mensajes_iguales"].sum())})
resumen = pd.DataFrame(resumen)
display(resumen)
todo_igual = (resumen["ventana_igual"] == resumen["llamadas"]).all() and (resumen["mensajes_iguales"] == resumen["llamadas"]).all()
print("Todos los mensajes son idénticos a los de la memoria" if todo_igual else "HAY DIFERENCIAS: revisar antes de seguir")

## 3. Entradas: ventanas y referencia del autor

Las mismas ventanas y la misma referencia que en la memoria: las 29 ventanas de desarrollo (validación) con tu etiqueta y las 40 de prueba con tu referencia y su grupo (alerta de consenso, alta actividad sin alerta, al azar). Se comprueba la huella de cada ventana contra los datos preparados. Tus notas no se copian.

**3.1 (una sola vez, solo el autor):** crea los dos ficheros en `privado/experimento_2/entradas/` a partir de los ficheros de trabajo, que solo se leen.

In [ ]:
# --- 3.1 Entradas (una sola vez): ventanas de desarrollo y de prueba con la referencia del autor ---
ORIGEN_H2 = "/content/drive/MyDrive/TFM_VIU/export_colab_split_estratificado/h2"   # solo lectura

if not os.path.exists(f"{RUTA_ENTRADAS}/ventanas_prueba.csv"):
    n_dev, n_prueba = exp2.copiar_entradas(
        fuente,
        origen_seleccion_dev=f"{ORIGEN_H2}/seleccion_ventanas_val_202610071447.json",
        origen_etiquetas_dev=f"{ORIGEN_H2}/prompt_engineering/etiquetas_desarrollo.csv",
        origen_referencia_prueba=f"{ORIGEN_H2}/fase2/referencia_fase2_d78659aafec5.csv",
        destino=RUTA_ENTRADAS)
    print(f"Entradas creadas: {n_dev} ventanas de desarrollo y {n_prueba} de prueba")
else:
    print("Las entradas ya existen: no se tocan")

In [ ]:
# --- 3.2 Carga de las entradas (con la huella de cada ventana comprobada de nuevo) ---
VENTANAS = {"dev": exp2.cargar_ventanas(f"{RUTA_ENTRADAS}/ventanas_desarrollo.csv", fuente),
            "test": exp2.cargar_ventanas(f"{RUTA_ENTRADAS}/ventanas_prueba.csv", fuente)}
for conjunto, ventanas in VENTANAS.items():
    tabla = pd.DataFrame(ventanas)
    print(f"{conjunto}: {len(tabla)} ventanas | referencia:", tabla["referencia"].value_counts().to_dict(),
          "| grupo:", tabla["grupo"].value_counts().to_dict())

## 4. Ejecuciones

- **4.1 (solo el autor):** copia (o actualiza, si han cambiado) las ejecuciones con otros modelos (Qwen2.5-14B y Qwen3.5-9B, en desarrollo y en prueba) y el registro de usos de la prueba a `privado/experimento_2/ejecuciones/`.
- **4.2 y 4.3:** solo para **repetir** una ejecución. Cada modelo hace 4 variantes × 29 ventanas en desarrollo (116 llamadas) o × 40 en prueba (160). Con Qwen3.5-9B en una T4, unos 30–60 min.

**Variantes:** las 4 de la prueba de la memoria (N0 y N1, con señales en filas y con reportes en filas). La versión 1 limita la respuesta a 300 tokens (la de la memoria, con Qwen2.5-7B); la 2, a 600, con los mismos mensajes (otros modelos).

In [ ]:
# --- 4.1 Copia (una sola vez) de las ejecuciones con otros modelos ---
ORIGEN_MODELOS = "/content/drive/MyDrive/TFM_VIU/export_colab_split_estratificado/experimento_2_modelos"   # solo lectura

os.makedirs(RUTA_EJECUCIONES, exist_ok=True)
copiados = 0
for ruta in sorted(glob.glob(f"{ORIGEN_MODELOS}/exp2_*.csv") + glob.glob(f"{ORIGEN_MODELOS}/exp2_*.json")
                   + glob.glob(f"{ORIGEN_MODELOS}/registro_uso_prueba.json")):
    destino = f"{RUTA_EJECUCIONES}/{os.path.basename(ruta)}"
    if not os.path.exists(destino) or os.path.getmtime(ruta) > os.path.getmtime(destino):   # nuevo o actualizado
        shutil.copy2(ruta, destino)
        copiados += 1
print(f"Copiados: {copiados}")
display(pd.DataFrame({"fichero": sorted(f for f in os.listdir(RUTA_EJECUCIONES) if f.endswith(".csv"))}))

In [ ]:
# --- 4.2 Parámetros de una ejecución nueva (EDITABLE; solo para repetir) ---
MODELO = "qwen3.5-9b"       # clave de llm_lab.modelos.CATALOGO_MODELOS
VERSION = 2                 # 1 = 300 tokens (memoria, Qwen2.5-7B); 2 = 600 tokens (otros modelos)
CONJUNTO = "dev"            # "dev" o "test"
CONFIRMO_PRUEBA = False     # obligatorio en True para "test"; cada uso queda registrado
REANUDAR = None             # nombre base de una ejecución interrumpida

VARIANTES = [f"{v}@v{VERSION}" for v in exp2.VARIANTES_PRUEBA]
print("Variantes:", VARIANTES)

In [ ]:
# --- 4.3 Ejecución (GPU) ---
import importlib.util
if importlib.util.find_spec("bitsandbytes") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes"])
    raise SystemExit("bitsandbytes instalado: reinicia la sesión y vuelve a ejecutar desde la sección 1")

from llm_lab.ejecucion import ejecutar
from llm_lab.modelos import ModeloLLM

modelo = ModeloLLM(MODELO)
BASE = ejecutar(prompts, modelo, fuente, VENTANAS[CONJUNTO], VARIANTES, RUTA_EJECUCIONES,
                prefijo=f"exp2_{CONJUNTO}", confirmo_prueba=CONFIRMO_PRUEBA, reanudar=REANUDAR)
modelo.liberar()